# Agent 2 — Dietary Agent (Package Edition)

This notebook is the **post-extraction demo** for Agent 2. All class and function definitions now live in the `immunosense.agents.dietary` package. This notebook imports them and runs the same end-to-end validations from the original `02_Dietrary_Exploration.ipynb`.

**Architecture (unchanged):**
- **Layer 1**: NHANES-based quantile regression models for population DII percentiles (trained offline via `python -m immunosense.agents.dietary.layer1_train`)
- **Layer 2**: Per-meal pipeline — extractor → food matcher → nutrient density → triggers + GL → daily rollup into 12-feature vector
- **Layer 3**: `DietaryAgent` — overnight fast tracker + per-feature robust baselines + multi-pathway permutation-test trigger detection

**The 10-dim output vector** (for Conductor + JEPA):
- Indices 0–5: anomaly scores for `dii_score`, `omega6_omega3_ratio`, `glycemic_load`, `sodium_mg`, `alcohol_g`, `overnight_fast_hours`
- Indices 6–9: trigger booleans `gluten_present`, `dairy_present`, `nightshade_present`, `upf_present` (cast to 0.0/1.0)

**Critical extraction notes:**
- Original notebook had 5 duplicated class definitions and 5 duplicated functions across cells. Only the *later* (live) definitions were extracted.
- DII is computed at the **daily** level only — not per-meal — because the Shivappa z-score transform is non-linear.
- The detector reports **raw permutation p-values** with no BH correction. The Conductor applies cross-agent FDR (per Challenge 3 architecture).

In [1]:
# === Package imports ===
import random as random_mod
from datetime import datetime

import numpy as np
import pandas as pd

from immunosense.agents.dietary import (
    DietaryAgent,
    process_meal,
    rollup_day,
    compute_intraday_gap_and_timestamps,
    compute_dii_row,
    compute_meal_dii,
    get_dii_percentile,
    MockExtractor,
    ClaudeHaikuExtractor,
    make_default_extractor,
    OvernightFastTracker,
    DietaryRobustTracker,
    DietaryTriggerDetector,
    classify_food_triggers,
    CONTINUOUS_FEATURES,
    BOOLEAN_TRIGGERS,
    DailyRollup,
    MealResult,
)
from immunosense.agents.dietary.constants import DII_REF

print(f'Imports OK. DII has {len(DII_REF)} components.')
print(f'Layer 3 features: {len(CONTINUOUS_FEATURES)} continuous + {len(BOOLEAN_TRIGGERS)} boolean')
print(f'DietaryAgent.output_dim = {DietaryAgent.output_dim}')

Imports OK. DII has 27 components.
Layer 3 features: 6 continuous + 4 boolean
DietaryAgent.output_dim = 10


In [2]:
# === Section 1: Per-food trigger classifier smoke test ===
# Verifies the trigger classification logic on a handful of representative foods.

test_foods = [
    (11111000, 'Milk, whole, 3.25% milkfat'),
    (51000000, 'Bread, white, toast'),
    (58101000, 'Rice, white, cooked'),       # gluten-free grain
    (74100000, 'Tomato, raw'),                # nightshade
    (74999000, 'Sweet potato, baked'),        # excluded from nightshade
    (92200000, 'Cola, soda, 12 fl oz'),       # UPF
    (24100000, 'Chicken breast, cooked'),     # protein, GI=0
]

for code, desc in test_foods:
    result = classify_food_triggers(code, desc)
    flags = [k for k, v in result.items() if k != 'estimated_gi' and v]
    print(f'  {desc[:35]:35s}  triggers={flags}  GI={result["estimated_gi"]}')

  Milk, whole, 3.25% milkfat           triggers=['dairy']  GI=31
  Bread, white, toast                  triggers=['gluten']  GI=70
  Rice, white, cooked                  triggers=[]  GI=73
  Tomato, raw                          triggers=['nightshade']  GI=15
  Sweet potato, baked                  triggers=[]  GI=78
  Cola, soda, 12 fl oz                 triggers=['upf']  GI=65
  Chicken breast, cooked               triggers=[]  GI=0


In [3]:
# === Section 2: Layer 3 Validation — Synthetic Patient ===
#
# Generates N days of dietary trajectories with KNOWN ground-truth triggers,
# then verifies the DietaryAgent detects them at the planted lag.

class SyntheticDietaryPatient:
    """Generates dietary trajectories with calibrated true-trigger relationships."""
    
    def __init__(self, triggers=None, flare_lag_days=2,
                 trigger_flare_probability=0.70,
                 baseline_flare_probability=0.10,
                 random_seed=42):
        self.triggers = triggers or []
        self.flare_lag_days = flare_lag_days
        self.trigger_flare_probability = trigger_flare_probability
        self.baseline_flare_probability = baseline_flare_probability
        self.rng = random_mod.Random(random_seed)
        self.np_rng = np.random.RandomState(random_seed)
    
    def _is_trigger_active(self, features):
        for trig in self.triggers:
            if trig in BOOLEAN_TRIGGERS:
                if features.get(trig, False):
                    return True
            elif trig in CONTINUOUS_FEATURES:
                val = features.get(trig, 0.0)
                if trig == 'sodium_mg' and val > 3000: return True
                elif trig == 'glycemic_load' and val > 120: return True
                elif trig == 'alcohol_g' and val > 15: return True
        return False
    
    def generate(self, n_days=60, start_date='2026-03-01'):
        records, scheduled_flares = [], {}
        start = pd.Timestamp(start_date)
        
        for i in range(n_days):
            date = (start + pd.Timedelta(days=i)).strftime('%Y-%m-%d')
            features = {
                'date': date,
                'dii_score':            float(self.np_rng.normal(0.5, 1.0)),
                'omega6_omega3_ratio':  float(max(1.0, self.np_rng.normal(10, 4))),
                'glycemic_load':        float(max(20, self.np_rng.normal(100, 30))),
                'sodium_mg':            float(max(500, self.np_rng.normal(2500, 800))),
                'alcohol_g':            float(max(0, self.np_rng.exponential(5))),
                'gluten_present':       self.rng.random() < 0.55,
                'dairy_present':        self.rng.random() < 0.40,
                'nightshade_present':   self.rng.random() < 0.30,
                'upf_present':          self.rng.random() < 0.25,
                'first_meal_timestamp': f'{date}T08:00:00',
                'last_meal_timestamp':  f'{date}T19:00:00',
            }
            
            flare_prob = (self.trigger_flare_probability
                          if self._is_trigger_active(features)
                          else self.baseline_flare_probability)
            
            if self.rng.random() < flare_prob:
                flare_date = (start + pd.Timedelta(days=i + self.flare_lag_days)).strftime('%Y-%m-%d')
                severity = float(self.np_rng.uniform(1.5, 3.0))
                scheduled_flares[flare_date] = max(scheduled_flares.get(flare_date, 0.0), severity)
            
            records.append(features)
        
        return records, scheduled_flares


def synthetic_record_to_rollup(rec):
    """Convert a synthetic record dict into a DailyRollup."""
    return DailyRollup(
        date=rec['date'],
        meal_count=4,
        daily_nutrients={},
        dii_score=rec['dii_score'],
        omega6_omega3_ratio=rec['omega6_omega3_ratio'],
        glycemic_load=rec['glycemic_load'],
        sodium_mg=rec['sodium_mg'],
        alcohol_g=rec['alcohol_g'],
        first_meal_timestamp=rec['first_meal_timestamp'],
        last_meal_timestamp=rec['last_meal_timestamp'],
        longest_intraday_gap_hours=float('nan'),
        gluten_present=rec['gluten_present'],
        dairy_present=rec['dairy_present'],
        nightshade_present=rec['nightshade_present'],
        upf_present=rec['upf_present'],
        feature_confidence={'dii_score': 'high'},
        daily_dii_percentile=None,
        meal_results=[],
    )


def run_validation_test(label, patient, expectation):
    records, flares = patient.generate(n_days=60)
    agent = DietaryAgent(window=14, n_permutations=500)
    for r in records:
        agent.observe(synthetic_record_to_rollup(r))
    for date, severity in flares.items():
        agent.observe_flare(date, severity)
    
    report = agent.analyze()
    print(f'\n{label}')
    print(f'  Expectation:  {expectation}')
    print(f'  Observed:     {report.n_days_observed} days, {report.n_flare_events} flare events')
    print(f'  Patterns ({len(report.detected_patterns)} found):')
    for p in report.detected_patterns:
        print(f'    {p.feature:35s} lag={p.lag_days}d  effect={p.effect_size:+.3f}'
              f'  p={p.p_value:.3f}  [{p.confidence}]')

print('SyntheticDietaryPatient + helpers defined.')

SyntheticDietaryPatient + helpers defined.


In [4]:
# === Section 3: Run the three validation tests ===
print('=' * 70)
print('Agent 2 Layer 3 validation - 3 tests')
print('=' * 70)

run_validation_test(
    'TEST 1: DAIRY trigger (boolean, lag=2 days, p=0.70)',
    SyntheticDietaryPatient(triggers=['dairy_present'], flare_lag_days=2,
                            trigger_flare_probability=0.70,
                            baseline_flare_probability=0.10, random_seed=42),
    'dairy_present at lag=2 with [high] or [medium] confidence',
)

run_validation_test(
    'TEST 2: NEGATIVE CONTROL (no triggers)',
    SyntheticDietaryPatient(triggers=[], baseline_flare_probability=0.15, random_seed=43),
    'Few or zero patterns at [high]; some at [medium]/[low] expected '
    '(~40 hypotheses × p<0.10 → ~4 false positives by chance — Conductor applies FDR)',
)

run_validation_test(
    'TEST 3: SODIUM trigger (continuous threshold, lag=1)',
    SyntheticDietaryPatient(triggers=['sodium_mg'], flare_lag_days=1,
                            trigger_flare_probability=0.70,
                            baseline_flare_probability=0.10, random_seed=44),
    'sodium_mg (>p75/p85) at lag=1 with [high] or [medium] confidence',
)

print('\n' + '=' * 70)
print('Layer 3 validation complete.')
print('=' * 70)

Agent 2 Layer 3 validation - 3 tests

TEST 1: DAIRY trigger (boolean, lag=2 days, p=0.70)
  Expectation:  dairy_present at lag=2 with [high] or [medium] confidence
  Observed:     60 days, 26 flare events
  Patterns (9 found):
    dairy_present                       lag=2d  effect=+1.520  p=0.000  [high]
    dii_score (>p85)                    lag=3d  effect=+0.798  p=0.058  [low]
    dii_score (>p90)                    lag=1d  effect=+0.780  p=0.072  [low]
    gluten_present                      lag=0d  effect=+0.760  p=0.010  [medium]
    alcohol_g (>p75)                    lag=1d  effect=+0.724  p=0.024  [medium]
    sodium_mg (>p90)                    lag=3d  effect=+0.686  p=0.094  [low]
    sodium_mg (>p75)                    lag=0d  effect=+0.667  p=0.018  [medium]
    omega6_omega3_ratio (>p75)          lag=2d  effect=+0.483  p=0.092  [low]
    dairy_present                       lag=0d  effect=+0.412  p=0.092  [low]

TEST 2: NEGATIVE CONTROL (no triggers)
  Expectation:  Few o

In [5]:
# === Section 4: BaseAgent contract — Conductor-facing output ===
#
# DietaryAgent.process({'rollup': rollup}) returns AgentOutput with the
# 10-dim vector, alerts, confidence, and trace_id for the Conductor.

patient = SyntheticDietaryPatient(triggers=['dairy_present'], flare_lag_days=2, random_seed=42)
records, flares = patient.generate(n_days=20)

agent = DietaryAgent(patient_id='demo_patient', window=14, n_permutations=500)
for r in records[:19]:
    agent.observe(synthetic_record_to_rollup(r))
for date, severity in flares.items():
    agent.observe_flare(date, severity)

# Process the 20th day via the BaseAgent interface
out = agent.process({'rollup': synthetic_record_to_rollup(records[19])})

print(f'agent_id:     {out.agent_id}')
print(f'vector_dim:   {out.vector_dim}')
print(f'vector:       {np.round(out.vector, 3)}')
print(f'confidence:   {out.confidence:.3f}')
print(f'trace_id:     {out.trace_id}')
print(f'alerts:       {len(out.alerts)} pattern alert(s)')
for a in out.alerts[:5]:
    print(f'  - {a["name"]:30s}  lag={a["lag_days"]}d  severity={a["severity"]}  p={a["p_value"]:.3f}')

health = agent.get_status()
print(f'\nAgent health: status={health.status}, latency={health.avg_latency_ms:.1f}ms, '
      f'errors={health.error_count_24hr}')

agent_id:     agent2_dietary
vector_dim:   10
vector:       [-5.3600e-01 -9.3300e-01  1.0000e-02  3.3800e-01  2.1453e+01  0.0000e+00
  1.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00]
confidence:   1.000
trace_id:     agent2_dietary-bd4d9241
alerts:       7 pattern alert(s)
  - dairy_present                   lag=2d  severity=high  p=0.002
  - omega6_omega3_ratio (>p85)      lag=2d  severity=medium  p=0.040
  - dii_score (>p85)                lag=3d  severity=medium  p=0.038
  - sodium_mg (>p75)                lag=0d  severity=high  p=0.006
  - dii_score (>p75)                lag=0d  severity=medium  p=0.028

Agent health: status=healthy, latency=452.9ms, errors=0


In [6]:
# === Section 5: Layer 1 + Layer 2 pipeline (only if NHANES artifacts present) ===
#
# This section runs the FULL pipeline (text -> meal -> rollup -> agent).
# Requires:
#   - ./artifacts/agent2_layer1/  (run: python -m immunosense.agents.dietary.layer1_train)
#   - ./artifacts/agent2_layer2/
#
# Skipped gracefully if artifacts don't exist (typical for first-time setup).

from pathlib import Path
from immunosense.agents.dietary.dii import load_layer1_artifacts
from immunosense.agents.dietary.density import (
    load_nutrient_density_cache,
    build_food_search_index,  # only used if rebuilding from NHANES
)
from immunosense.agents.dietary.matching import (
    FoodMatcher,
    load_food_search_index,
)

layer1_dir = Path('./artifacts/agent2_layer1')
layer2_dir = Path('./artifacts/agent2_layer2')
density_cache = layer2_dir / 'nutrient_density_per_100g_v2.pkl'
food_index_cache = layer2_dir / 'food_code_search_index.pkl'

if not (layer1_dir.exists() and density_cache.exists() and food_index_cache.exists()):
    print('Layer 1/2 artifacts not present — skipping live pipeline demo.')
    print('To build them, run from a terminal:')
    print('  python -m immunosense.agents.dietary.layer1_train')
else:
    # Load all artifacts
    layer1 = load_layer1_artifacts(layer1_dir)
    density_df = load_nutrient_density_cache(density_cache)
    food_index = load_food_search_index(food_index_cache)
    
    density_by_code = density_df.set_index('food_code')
    matcher = FoodMatcher(food_index)
    
    extractor = MockExtractor()  # use Claude if ANTHROPIC_API_KEY is set
    
    print(f'Layer 1: {len(layer1["dii_models"])} quantile models loaded')
    print(f'Layer 2: {len(density_df):,} food codes with nutrient data')
    print(f'Layer 2: {len(food_index):,} food codes in search index')
    
    # Process 4 meals -> daily rollup -> percentile lookup
    patient_demo = {'age': 45, 'sex': 2, 'bmi': 28}
    meals = [
        process_meal('Two scrambled eggs and toast', extractor=extractor,
                     food_matcher=matcher, density_by_code=density_by_code,
                     timestamp='2026-05-22T08:00:00'),
        process_meal('Chicken with rice and salad', extractor=extractor,
                     food_matcher=matcher, density_by_code=density_by_code,
                     timestamp='2026-05-22T12:30:00'),
        process_meal('An apple as a snack', extractor=extractor,
                     food_matcher=matcher, density_by_code=density_by_code,
                     timestamp='2026-05-22T15:00:00'),
        process_meal('Salmon with rice, avocado on the side', extractor=extractor,
                     food_matcher=matcher, density_by_code=density_by_code,
                     timestamp='2026-05-22T19:00:00'),
    ]
    
    day = rollup_day(meals, date='2026-05-22',
                     age=patient_demo['age'], sex=patient_demo['sex'], bmi=patient_demo['bmi'],
                     dii_models=layer1['dii_models'], feature_cols=layer1['feature_cols'])
    
    print(f'\nPatient: {patient_demo} | Date: {day.date} | meals: {day.meal_count}')
    print(f'  dii_score:                   {day.dii_score:+.2f}')
    print(f'  dii_percentile:              {day.daily_dii_percentile:.0%}' if day.daily_dii_percentile else '  dii_percentile:              n/a')
    print(f'  omega6_omega3_ratio:         {day.omega6_omega3_ratio:.2f}')
    print(f'  glycemic_load:               {day.glycemic_load:.1f}')
    print(f'  sodium_mg:                   {day.sodium_mg:.0f}')
    print(f'  alcohol_g:                   {day.alcohol_g:.1f}')
    print(f'  triggers:  gluten={day.gluten_present} dairy={day.dairy_present} '
          f'nightshade={day.nightshade_present} upf={day.upf_present}')

Layer 1: 7 quantile models loaded
Layer 2: 5,254 food codes with nutrient data
Layer 2: 5,254 food codes in search index

Patient: {'age': 45, 'sex': 2, 'bmi': 28} | Date: 2026-05-22 | meals: 4
  dii_score:                   +1.16
  dii_percentile:              49%
  omega6_omega3_ratio:         3.99
  glycemic_load:               101.8
  sodium_mg:                   2178
  alcohol_g:                   0.0
  triggers:  gluten=True dairy=True nightshade=True upf=False


## ✅ Migration complete

**What was extracted:**
- 22 classes (5 duplicates removed) and 21 functions (5 duplicates removed) from the source 2,315-line notebook
- 14 modules under `immunosense.agents.dietary` (constants, types, sources/, matching, density, dii, triggers, pipeline, trackers, detector, agent, layer1_train)
- 133 unit + integration tests passing in ~5 seconds

**What this notebook validates end-to-end:**
- Per-food trigger classification (dairy/gluten/nightshade/UPF + GI)
- Layer 3 trigger detection with synthetic patients (3 planted scenarios)
- BaseAgent contract (10-dim vector, alerts, health)
- Full Layer 1 + Layer 2 pipeline (when NHANES artifacts are present)

**Next sprint:** Sprint 4 will extract Agent 1 (Biomarker).